In [2]:
#importing important libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

In [3]:
#loading daatset
df = pd.read_csv("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
print(df)

         Destination Port   Flow Duration   Total Fwd Packets  \
0                   54865               3                   2   
1                   55054             109                   1   
2                   55055              52                   1   
3                   46236              34                   1   
4                   54863               3                   2   
...                   ...             ...                 ...   
225740              61374              61                   1   
225741              61378              72                   1   
225742              61375              75                   1   
225743              61323              48                   2   
225744              61326              68                   1   

         Total Backward Packets  Total Length of Fwd Packets  \
0                             0                           12   
1                             1                            6   
2                          

In [4]:
#striping column names
df.columns = df.columns.str.strip()

In [5]:
#Deleting constant columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print("\nDeleting constant columns:", constant_cols)

df.drop(columns=constant_cols, inplace=True)



Deleting constant columns: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [6]:
#replacing infinite value with NaN
df = df.replace([np.inf, -np.inf], np.nan)

In [7]:
#seprating categoricak and numerical value
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

In [8]:
#filling missing values for numerical column
skew_values = df[num_cols].skew()

for col in num_cols:
    if abs(skew_values[col]) <= 0.5:
        # symmetrical → fill with mean
        df[col] = df[col].fillna(df[col].mean())
    else:
        # skewed → fill with median
        df[col] = df[col].fillna(df[col].median())

In [9]:

#filling missing values for categorical column
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
#Encode Target Value

label_encoder = LabelEncoder()
df["Label_enc"] = label_encoder.fit_transform(df["Label"])

y = df["Label_enc"]
X = df.drop(columns=["Label", "Label_enc"])

In [11]:
#Encoding categorical columns
for col in X.select_dtypes(include=["object"]).columns:
    X[col] = LabelEncoder().fit_transform(X[col])


In [12]:
#Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [13]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)


In [14]:
#SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [15]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": LinearSVC(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [16]:
#Training and Evaluation of All Models
'''for name, model in models.items():
    print(f"\n================ {name} ================")

    # Train
    model.fit(X_train_res, y_train_res)

    # Predictions
    train_pred = model.predict(X_train_res)
    test_pred = model.predict(X_test)

    # Metrics
    train_acc = accuracy_score(y_train_res, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)

    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred))'''


'for name, model in models.items():\n    print(f"\n================ {name} ================")\n\n    # Train\n    model.fit(X_train_res, y_train_res)\n\n    # Predictions\n    train_pred = model.predict(X_train_res)\n    test_pred = model.predict(X_test)\n\n    # Metrics\n    train_acc = accuracy_score(y_train_res, train_pred)\n    test_acc = accuracy_score(y_test, test_pred)\n    f1 = f1_score(y_test, test_pred)\n\n    print(f"Training Accuracy: {train_acc:.4f}")\n    print(f"Test Accuracy: {test_acc:.4f}")\n    print(f"F1 Score: {f1:.4f}")\n    print("\nClassification Report:")\n    print(classification_report(y_test, test_pred))'

In [17]:
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# Prepare lists for final tables
train_rows = []
test_rows = []

for name, model in models.items():
    print(f"\n================ {name} ================")

    # Train model
    model.fit(X_train_res, y_train_res)

    # Predictions
    train_pred = model.predict(X_train_res)
    test_pred = model.predict(X_test)

    # Probabilities for AUC
    if hasattr(model, "predict_proba"):
        y_test_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_test_proba = model.decision_function(X_test)
    else:
        y_test_proba = None

    # TRAIN metrics
    train_acc = accuracy_score(y_train_res, train_pred)
    train_sens = recall_score(y_train_res, train_pred)
    train_spec = specificity_score(y_train_res, train_pred)
    train_f1 = f1_score(y_train_res, train_pred)

    # TEST metrics
    test_acc = accuracy_score(y_test, test_pred)
    test_sens = recall_score(y_test, test_pred)
    test_spec = specificity_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred)
    test_auc = roc_auc_score(y_test, y_test_proba) if y_test_proba is not None else None

    # Append to lists
    train_rows.append([name, train_acc, train_sens, train_spec, train_f1, None])
    test_rows.append([name, test_acc, test_sens, test_spec, test_f1, test_auc])

# Convert to DataFrame
train_df = pd.DataFrame(train_rows, columns=["Model", "Accuracy", "Sensitivity", "Specificity", "F1 Score", "AUC"])
test_df = pd.DataFrame(test_rows, columns=["Model", "Accuracy", "Sensitivity", "Specificity", "F1 Score", "AUC"])

# Display Tables
print("\n\n========== TRAIN METRICS ==========")
print(train_df.to_string(index=False))

print("\n\n========== TEST METRICS ==========")
print(test_df.to_string(index=False))



================ Logistic Regression ================

================ Random Forest ================

================ Decision Tree ================

================ SVM ================

================ Gradient Boosting ================


========== TRAIN METRICS ==========
              Model  Accuracy  Sensitivity  Specificity  F1 Score  AUC
Logistic Regression  0.998618     0.998594     0.998643  0.998618 None
      Random Forest  1.000000     1.000000     1.000000  1.000000 None
      Decision Tree  1.000000     1.000000     1.000000  1.000000 None
                SVM  0.999155     0.998906     0.999404  0.999155 None
  Gradient Boosting  0.999863     0.999775     0.999951  0.999863 None


========== TEST METRICS ==========
              Model  Accuracy  Sensitivity  Specificity  F1 Score      AUC
Logistic Regression  0.998693     0.998750     0.998619  0.998848 0.999849
      Random Forest  0.999934     0.999922     0.999949  0.999941 0.999980
      Decision Tree  0.999934